# wrap-forward-fn-generic composite — cx1: wire a new op into autograd — wrap + register + signature

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `wrap-forward-fn-generic`, `register-back-fn-after-wrap`, `backward-fn-signature`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "wrap-forward-fn-generic"
DD_ATOM_IDS = ["wrap-forward-fn-generic", "register-back-fn-after-wrap", "backward-fn-signature"]
DD_SUBTOPICS = ["Backprop: wrap forward fn", "Backprop: register back fn", "Backprop: backward fn signature"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Wiring a new op into autograd — the trifecta

Three atoms compose into one workflow whenever ARENA adds a new differentiable op:

1. **`wrap-forward-fn-generic`** — turn a raw numerical fn (`torch.log`) into a
   `Tensor`-aware fn by unboxing args, calling the raw fn, then boxing the result
   inside a fresh closure.
2. **`backward-fn-signature`** — write a back fn that follows the uniform
   `(grad_out, out, *args) -> grad_in` contract so the dispatcher can call it
   generically. For elementwise ops the body is `grad_out * local_derivative`.
3. **`register-back-fn-after-wrap`** — store the back fn in a lookup table keyed
   by `(forward_fn, argnum)` so the reverse pass can find it by tuple key.

Composition: the order is fixed — wrap, then write back fn with canonical
signature, then register under the `(fwd_fn, argnum)` key. Skipping any one
step leaves the op invisible to the dispatcher.


### Composite Exercise — wire a new op into autograd — wrap + register + signature

**Atoms exercised together**: `wrap-forward-fn-generic`, `register-back-fn-after-wrap`, `backward-fn-signature`

Wire `torch.log` end-to-end into the tiny autograd. Build a single function
`cx1_wire_log()` that returns a 3-tuple `(tlog, BACK_FUNCS, log_back)` where:

- `tlog` is the `Tensor`-aware wrapper for `torch.log`, produced via your
  `wrap_forward_fn(t.log)`. Calling `tlog(Tensor([1., e, e**2]))` must return
  a `Tensor` whose `.array` is `[0., 1., 2.]`.
- `BACK_FUNCS` is a fresh `BackwardFuncLookup` with `log_back` registered at
  `(torch.log, 0)`.
- `log_back(grad_out, out, x)` follows the canonical back-fn signature and
  returns `grad_out / x`.

The test then exercises the trifecta: it (a) calls `tlog` on a `Tensor`,
(b) looks up `(torch.log, 0)` from the table, and (c) invokes the looked-up
back fn through that dispatch path.


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class Tensor:
    """Thin wrapper around a raw torch.Tensor stored on .array."""
    def __init__(self, array):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
    def __repr__(self):
        return f'Tensor({self.array.tolist()})'


class BackwardFuncLookup:
    def __init__(self):
        raise NotImplementedError()
    def add_back_func(self, fwd_fn, argnum, back_fn):
        raise NotImplementedError()
    def get_back_func(self, fwd_fn, argnum):
        raise NotImplementedError()


def wrap_forward_fn(fwd_fn):
    """Return a Tensor-aware wrapper: unbox args, call fwd_fn, box result."""
    raise NotImplementedError()


def cx1_wire_log():
    """Return (tlog, BACK_FUNCS, log_back) — all three atoms composed."""
    raise NotImplementedError()


def _test_cx1():
    import math
    tlog, BACK_FUNCS, log_back = cx1_wire_log()

    # (1) wrap_forward_fn produced a Tensor-aware wrapper for torch.log
    a = Tensor(t.tensor([1.0, math.e, math.e ** 2]))
    b = tlog(a)
    assert isinstance(b, Tensor), f'tlog must return a Tensor, got {type(b)}'
    assert isinstance(b.array, t.Tensor), '.array must be a torch.Tensor'
    assert t.allclose(b.array, t.tensor([0.0, 1.0, 2.0]), atol=1e-5), f'tlog value: {b.array}'

    # (2) BACK_FUNCS holds log_back at the (torch.log, 0) key — dispatch path works
    fn = BACK_FUNCS.get_back_func(t.log, 0)
    assert fn is log_back, f'dispatch returned {fn}, expected the same log_back fn object'

    # (3) canonical signature: log_back(grad_out, out, x) = grad_out / x
    x = t.tensor([1.0, 2.0, 4.0])
    out = t.log(x)
    g = log_back(t.ones(3), out, x)
    assert g.shape == x.shape
    assert t.allclose(g, t.tensor([1.0, 0.5, 0.25])), f'log_back value: {g}'

    # (4) end-to-end: call the looked-up back fn through the table
    fn_again = BACK_FUNCS.get_back_func(t.log, 0)
    g2 = fn_again(t.tensor([3.0, -2.0, 10.0]), out, x)
    assert t.allclose(g2, t.tensor([3.0, -1.0, 2.5])), f'dispatched call: {g2}'

    _dd_passed.add('cx1')

_test_cx1()

<details><summary>Show solution — cx1</summary>

```python
class Tensor:
    def __init__(self, array):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
    def __repr__(self):
        return f'Tensor({self.array.tolist()})'

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        return Tensor(fwd_fn(*raw, **kwargs))
    return tensor_func

def cx1_wire_log():
    # atom 1: wrap the forward fn into a Tensor-aware closure
    tlog = wrap_forward_fn(t.log)
    # atom 2: write a back fn with the canonical (grad_out, out, x) signature
    def log_back(grad_out, out, x):
        return grad_out / x
    # atom 3: register it under the (fwd_fn, argnum) key after wrapping
    BACK_FUNCS = BackwardFuncLookup()
    BACK_FUNCS.add_back_func(t.log, 0, log_back)
    return tlog, BACK_FUNCS, log_back

```

All three atoms compose in one function: `tlog = wrap_forward_fn(...)` is the
wrap atom, the `def log_back(grad_out, out, x): return grad_out / x` line is the
signature-contract atom, and `BACK_FUNCS.add_back_func(t.log, 0, log_back)` is the
register atom. The test dispatches through `BACK_FUNCS.get_back_func(t.log, 0)` —
if any one of the three steps is wrong, dispatch fails.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx1',
        'subtopics': ["Backprop: wrap forward fn", "Backprop: register back fn", "Backprop: backward fn signature"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()